In [ ]:
%pip install kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ashydv/advertising-dataset")

print("Path to dataset files:", path)

In [1]:
import pandas as pd

df = pd.read_csv('advertising.csv')


In [2]:
df.head(5)

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9


In [3]:
df.describe()

,TV,Radio,Newspaper,Sales
count,200.000000,200.000000,200.000000,200.000000
mean,147.042500,23.264000,30.554000,15.130500
std,85.854236,14.846809,21.778621,5.283892
min,0.700000,0.000000,0.300000,1.600000
25%,74.375000,9.975000,12.750000,11.000000
50%,149.750000,22.900000,25.750000,16.000000
75%,218.825000,36.525000,45.100000,19.050000
max,296.400000,49.600000,114.000000,27.000000


In [4]:
y = df.Sales

In [5]:
X = df.drop(columns=['Sales'])

In [6]:
#split the dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression


X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42 )

reg = LinearRegression()


In [7]:
#try minmax scaler 
from sklearn.preprocessing import MinMaxScaler

reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)
residuals = y_test - y_pred

#Mean Squared error
for feature, coef in zip(X.columns, reg.coef_):
    print(f"{feature}: {coef:.4f}")
print("Test R²:", reg.score(X_test, y_test))

TV: 0.0545
Radio: 0.1009
Newspaper: 0.0043
Test R²: 0.9059011844150826


In [8]:
#try minmax scaler 
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Optional: convert back to DataFrame
X_train_scaled = pd.DataFrame( X_train_scaled,
                                columns=X.columns,
                                index=X_train.index
                            )
X_test_scaled = pd.DataFrame( X_test_scaled,
                              columns=X.columns,
                              index=X_test.index
                            )

reg.fit(X_train_scaled, y_train)
y_pred = reg.predict(X_test_scaled)

#Mean Squared error
for feature, coef in zip(X.columns, reg.coef_):
    print(f"{feature}: {coef:.4f}")
print("Test R²:", reg.score(X_test_scaled, y_test))


TV: 16.1184
Radio: 5.0069
Newspaper: 0.4363
Test R²: 0.9059011844150826


In [9]:
#try standard scaler

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled,
                              columns=X.columns,
                              index=X_train.index
                            )

X_test_scaled = pd.DataFrame( X_test_scaled,
                              columns=X.columns,
                              index=X_test.index
                            )

reg.fit(X_train_scaled, y_train)
y_pred = reg.predict( X_test_scaled)
residuals = y_test - y_pred

#Mean Squared error
for feature, coef in zip(X.columns, reg.coef_):
    print(f"{feature}: {coef:.4f}")
print("Test R²:", reg.score(X_test_scaled, y_test))


TV: 4.5872
Radio: 1.4898
Newspaper: 0.0879
Test R²: 0.9059011844150826


In [10]:
X_train_scaled.columns

Index(['TV', 'Radio', 'Newspaper'], dtype='object')

In [11]:
#check after creating the interaction term 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled,
                              columns=X.columns,
                              index=X_train.index
                            )

X_test_scaled = pd.DataFrame( X_test_scaled,
                              columns=X.columns,
                              index=X_test.index
                            )

X_train_scaled['TV_Radio'] = X_train_scaled['TV'] * X_train_scaled['Radio']
X_test_scaled['TV_Radio']  = X_test_scaled['TV'] * X_test_scaled['Radio']


In [12]:
X_train_scaled.columns

Index(['TV', 'Radio', 'Newspaper', 'TV_Radio'], dtype='object')

In [13]:
X_test_scaled.columns

Index(['TV', 'Radio', 'Newspaper', 'TV_Radio'], dtype='object')

In [14]:
reg = LinearRegression()


reg.fit(X_train_scaled, y_train)
y_pred = reg.predict( X_test_scaled)
residuals = y_test - y_pred

#Mean Squared error
for feature, coef in zip(X_test_scaled.columns, reg.coef_):
    print(f"{feature}: {coef:.4f}")
print("Test R²:", reg.score(X_test_scaled, y_test))


TV: 4.5519
Radio: 1.5027
Newspaper: 0.0813
TV_Radio: 0.4982
Test R²: 0.9229102666046112


In [15]:

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV

pipe = make_pipeline(
    StandardScaler(),                                              # center first → low-collinearity products
    PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
    StandardScaler(),                                              # re-scale so Lasso penalizes products fairly
    LassoCV(cv=5, max_iter=10000, random_state=42),
)


# X_train now has the new column, e.g. TV, Radio, Newspaper, Online
pipe.fit(X_train, y_train)


Pipeline(steps=[('standardscaler-1', StandardScaler()),
                ('polynomialfeatures',
                 PolynomialFeatures(include_bias=False, interaction_only=True)),
                ('standardscaler-2', StandardScaler()),
                ('lassocv', LassoCV(cv=5, max_iter=10000, random_state=42))])

In [16]:
#pipe.fit(X_train, y_train)

In [17]:
names = pipe[:-1].get_feature_names_out()
for n, c in zip(names, pipe[-1].coef_):
    print(f"{n}: {c:.4f}")
print("alpha:", pipe[-1].alpha_)
print("Test R²:", pipe.score(X_test, y_test))

TV: 4.4564
Radio: 1.4280
Newspaper: 0.0047
TV Radio: 0.4136
TV Newspaper: 0.0000
Radio Newspaper: 0.0000
alpha: 0.10786334798890901
Test R²: 0.9197401007818391


In [18]:
#check after creating the interaction term 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



X_train_scaled = pd.DataFrame(X_train_scaled,
                              columns=X.columns,
                              index=X_train.index
                            )

X_test_scaled = pd.DataFrame( X_test_scaled,
                              columns=X.columns,
                              index=X_test.index
                            )

X_train_scaled.drop( columns=['Newspaper'], inplace=True)
X_test_scaled.drop( columns=['Newspaper'], inplace=True)

X_train_scaled['TV_Radio'] = X_train_scaled['TV'] * X_train_scaled['Radio']
X_test_scaled['TV_Radio']  = X_test_scaled['TV'] * X_test_scaled['Radio']


In [19]:
#Mean Squared error
reg = LinearRegression()


reg.fit(X_train_scaled, y_train)
y_pred = reg.predict( X_test_scaled)
residuals = y_test - y_pred
for feature, coef in zip(X_train_scaled.columns, reg.coef_):
    print(f"{feature}: {coef:.4f}")
print("Test R²:", reg.score(X_test_scaled, y_test))


TV: 4.5517
Radio: 1.5343
TV_Radio: 0.4991
Test R²: 0.9241446091056417


In [21]:

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import RidgeCV

pipe = make_pipeline(
    StandardScaler(),                                              # center first → low-collinearity products
    PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
    StandardScaler(),                                              # re-scale so Lasso penalizes products fairly
    RidgeCV(cv=5)
)


# X_train now has the new column, e.g. TV, Radio, Newspaper, Online
pipe.fit(X_train, y_train)


Pipeline(steps=[('standardscaler-1', StandardScaler()),
                ('polynomialfeatures',
                 PolynomialFeatures(include_bias=False, interaction_only=True)),
                ('standardscaler-2', StandardScaler()),
                ('ridgecv', RidgeCV(cv=5))])

In [22]:
names = pipe[:-1].get_feature_names_out()
for n, c in zip(names, pipe[-1].coef_):
    print(f"{n}: {c:.4f}")
print("alpha:", pipe[-1].alpha_)
print("Test R²:", pipe.score(X_test, y_test))

TV: 4.5262
Radio: 1.4934
Newspaper: 0.0624
TV Radio: 0.5329
TV Newspaper: -0.0470
Radio Newspaper: 0.0718
alpha: 1.0
Test R²: 0.9241915586854306
